In [51]:
# import libraries
import pandas as pd
import numpy as np

In [52]:
# create a variable to store the file path
filepath_istat_data = '../raw/dati_istat.csv'

# create df_istat_data to read the Istat data from the file path
df_istat_data = pd.read_csv(filepath_istat_data)

# create a variable to store the file path for the Comuni data
filepath_comuni = '../raw/Comuni - Dimensione Data Indagine 23-06-2026 Stampa 23062026145519.csv'

# create df_comuni to read the Comuni data from the file path
df_comuni = pd.read_csv(filepath_comuni, sep=';')


In [53]:
# check for missing values in df_istat_data
df_istat_data.isnull().sum()

# delete columns with all missing values
df_istat_data.dropna(axis=1, how='all', inplace=True)

In [54]:
# check for missing values in the df_comuni
df_comuni.isnull().sum()

# keep only the columns that are needed for the analysis
df_comuni = df_comuni[['Codice Comune (numerico)', 'Comune', 'Superficie (Kmq)', 'Popolazione residente']]

# display the first 5 rows of the dataframe
df_comuni.head()

,Codice Comune (numerico),Comune,Superficie (Kmq),Popolazione residente
0,1001,Agliè,"13,1463",2585
1,1002,Airasca,"15,7393",3695
2,1003,Ala di Stura,"46,3316",463
3,1004,Albiano d'Ivrea,"11,7397",1624
4,1006,Almese,"17,8741",6297


In [55]:
# check the info of the dataframe
df_comuni.info()

# convert the 'Superficie (Kmq)' column to float
df_comuni['Superficie (Kmq)'] = df_comuni['Superficie (Kmq)'].str.replace('.', '', regex=False).str.replace(',', '.').astype(float)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Codice Comune (numerico)  7894 non-null   int64 
 1   Comune                    7893 non-null   object
 2   Superficie (Kmq)          7894 non-null   object
 3   Popolazione residente     7894 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 246.8+ KB


In [56]:
# check the info of the dataframe
df_comuni.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Codice Comune (numerico)  7894 non-null   int64  
 1   Comune                    7893 non-null   object 
 2   Superficie (Kmq)          7894 non-null   float64
 3   Popolazione residente     7894 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 246.8+ KB


In [57]:
# check for missing values in the 'Comune' column
df_comuni[df_comuni['Comune'].isnull()]

# replace the missing values in the 'Comune' column with 'None'
df_comuni.loc[df_comuni['Codice Comune (numerico)'] == 1168, 'Comune'] = 'None'

# check if the missing values in the 'Comune' column have been replaced
df_comuni[df_comuni['Codice Comune (numerico)'] == 1168]

,Codice Comune (numerico),Comune,Superficie (Kmq),Popolazione residente
164,1168,None,24.6422,7662


In [58]:
# create a new dataframe df_killinj that contains the sum of the 'OBS_VALUE' column for each 'REF_AREA' and 'TIME_PERIOD' where 'DATA_TYPE' is 'KILLINJ'
df_killinj=df_istat_data[df_istat_data['DATA_TYPE']=='KILLINJ'].groupby(['REF_AREA', 'TIME_PERIOD'])['OBS_VALUE'].sum().reset_index()

# create a new dataframe df_roadacc that contains the sum of the 'OBS_VALUE' column for each 'REF_AREA' and 'TIME_PERIOD' where 'DATA_TYPE' is 'ROADACC' and 'RESULT' is '9'
df_roadacc = df_istat_data[(df_istat_data['DATA_TYPE'] == 'ROADACC') & (df_istat_data['RESULT'] == '9')].groupby(['REF_AREA', 'TIME_PERIOD'])['OBS_VALUE'].sum().reset_index()

In [59]:
# merge the two dataframes on 'REF_AREA' and 'TIME_PERIOD'
df_merged_istat_data = pd.merge(df_killinj, df_roadacc, on=['REF_AREA', 'TIME_PERIOD'])

# rename the columns for clarity
df_merged_istat_data.rename(columns={'OBS_VALUE_x': 'KILLINJ', 'OBS_VALUE_y': 'ROADACC'}, inplace=True)

# display the first 5 rows of the merged dataframe
df_merged_istat_data.head()

,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC
0,1001,2001,10,5
1,1001,2002,10,5
2,1001,2003,7,4
3,1001,2004,13,9
4,1001,2005,2,2


In [60]:
# merge the df_merged_istat_data with df_comuni on 'REF_AREA' and 'Codice Comune (numerico)' to get the name of the comune
df_merged_istat_data_comuni_name=pd.merge(df_merged_istat_data, df_comuni, left_on='REF_AREA', right_on='Codice Comune (numerico)', how='left')

# display the first 5 rows of the merged dataframe
df_merged_istat_data_comuni_name.head()

,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC,Codice Comune (numerico),Comune,Superficie (Kmq),Popolazione residente
0,1001,2001,10,5,1001.0,Agliè,13.1463,2585.0
1,1001,2002,10,5,1001.0,Agliè,13.1463,2585.0
2,1001,2003,7,4,1001.0,Agliè,13.1463,2585.0
3,1001,2004,13,9,1001.0,Agliè,13.1463,2585.0
4,1001,2005,2,2,1001.0,Agliè,13.1463,2585.0


In [61]:
# delete the 'Codice Comune (numerico)' column as it is no longer needed
df_merged_istat_data_comuni_name.drop(columns=['Codice Comune (numerico)'], inplace=True)

# display the first 5 rows of the final dataframe
df_merged_istat_data_comuni_name.head()

,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC,Comune,Superficie (Kmq),Popolazione residente
0,1001,2001,10,5,Agliè,13.1463,2585.0
1,1001,2002,10,5,Agliè,13.1463,2585.0
2,1001,2003,7,4,Agliè,13.1463,2585.0
3,1001,2004,13,9,Agliè,13.1463,2585.0
4,1001,2005,2,2,Agliè,13.1463,2585.0
